# How to work with large data in Pandas

Source for small file at https://www.kaggle.com/datasets/adelanseur/taxi-trips-chicago-2024  
Source for the full 14 Gig, 212 MILLION rows at https://data.cityofchicago.org/Transportation/Taxi-Trips-2013-2023-/wrvz-psew/about_data  

This dataset contain over a hundred million rows Chicago taxi data.

**IMPORTANT NOTE** This notebook creates a number of files on your disk. It will be worth your time to revisit the `FILE_PATH` directory after this lecture and remove all unnecessary files.

In [ ]:
import pandas as pd

In [ ]:
FILE_PATH = '../../datasets/taxi-trips/'

COMPRESSED_FILE        = FILE_PATH  + 'taxi_trips_kaggle.zip'
UNCOMPRESSED_FILE      = FILE_PATH  + 'Taxi_Trips_-_2024_20240408.csv'

UNCOMPRESSED_TINY_FILE = FILE_PATH  + 'Taxi_Trips_-_2024_20240408_tiny.csv'
UNCOMPRESSED_4X_FILE   = FILE_PATH  + 'Taxi_Trips_-_2024_20240408_4x.csv'

FEATHER_FILE           = FILE_PATH  + 'taxi_trips.feather'
PARQUET_FILE           = FILE_PATH  + 'taxi_trips.parquet'

FEATHER_4X_FILE           = FILE_PATH  + 'taxi_trips.feather'
PARQUET_4X_FILE           = FILE_PATH  + 'taxi_trips.parquet'

### Make sure uncompressed file also exists

In [ ]:
import zipfile
with zipfile.ZipFile(COMPRESSED_FILE, 'r') as zip_ref:
    zip_ref.extractall(FILE_PATH)

#### Make a version that is 4 times as large
(to simulate a bigger file)

In [ ]:
# This cell may take a minute to run

# Inflation factor
INFLATION_FACTOR = 4

uncompressed_df = pd.read_csv(UNCOMPRESSED_FILE)

uncompressed_df.to_csv(UNCOMPRESSED_4X_FILE, index=False)

for _ in range(INFLATION_FACTOR - 1):
    uncompressed_df.to_csv(UNCOMPRESSED_4X_FILE, mode='a', header=False, index=False)

# Notice that this dataframe is being "unloaded" from memory to free up space
del uncompressed_df

### Also create a 100 line file for comparison

In [ ]:
tiny_df = pd.read_csv(UNCOMPRESSED_FILE, nrows=100)
tiny_df.to_csv(UNCOMPRESSED_TINY_FILE, index=False)


### What's in this file?

In [ ]:
tiny_df.head()

### Let's look at column types
(you will see why this is important)

In [ ]:
tiny_df.info()

### Let's load the 4x file and see how long it takes

In [ ]:
%%time
# Notice that we are not assigning the result to a variable. This is because we are just timing how long it takes to load the file.
pd.read_csv(UNCOMPRESSED_4X_FILE).head(3)

## Detour: what is actually happening when you load the file in Pandas?
When you can `read_csv`, Pandas is:
- loading the whole file into memory
- using the the first chunk of a few thousand rows to infer the column types
- doing any other processing you ask (such as datetime parsing, removing columns, limiting rows, etc.)

If the file is bigger than the amount of free memory on your computer, expect a crash!

## Use compressed files in low disk space environments
Pandas can read files compressed with common compression schemes. Often csv files can be compressed to 10% of their original size. This is a great way to save disk space. However, it will take longer to read the file because it has to be decompressed first. If you have a lot of disk space, it is better to use uncompressed files.

In [ ]:
COMPRESSED_FILE

In [ ]:
UNCOMPRESSED_FILE

In [ ]:
%timeit pd.read_csv(COMPRESSED_FILE).head(3)
%timeit pd.read_csv(UNCOMPRESSED_FILE).head(3)

How much space do they take up on disk? The compressed file is much smaller!

In [ ]:
# Mac/Linux
!ls -ltrhc $COMPRESSED_FILE

# Windows
#!dir $COMPRESSED_FILE

In [ ]:
# Mac/Linux
!ls -ltrhc $UNCOMPRESSED_FILE

# Windows
#!dir $UNCOMPRESSED_FILE

## Detour: Rows wise vs column wise data

Here is a sample markdown table with a few rows and columns of csv style data:

| Name | Age | City |
|------|-----|------|
| Alice | 30 | New York |
| Bob | 25 | Los Angeles |
| Carol | 35 | Chicago |
| Dave | 40 | Houston |
| Eve | 28 | Phoenix |
| Frank | 32 | Philadelphia |
| Grace | 29 | San Antonio |
| Henry | 33 | Dallas |


Recall that you can think of disk or memory as a gigantic array.  
```
[Alic][e, 3][0,Ne][w Yo][rk;B][ob,2][5,Lo][s An][gele][s;Ca]...
```

Say you only want to collect names of people in the table. In a text file such as `csv` format or even a standard database, you will read the name Alice (and save it), read the age 30 (but discard it), read the city New York (but discard it), read the name Bob (and save it), read the age 25 (but discard it), read the city Los Angeles (but discard it), and so on. This is called row wise data access. You are reading all the data, but only saving a small part of it.

```
[Alic][e,**][*,**][****][**;B][ob,*][*,**][****][****][*;Ca]...
```


What if you stored the same data in a columnar format as such:


|  |  |  |  |  |  |  |  |  |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |  
| **Name** | Alice | Bob | Carol | Dave | Eve | Frank | Grace | Henry |
| **Age** | 30 | 25 | 35 | 40 | 28 | 32 | 29 | 33 |
| **City** | New York | Los Angeles | Chicago | Houston | Phoenix | Philadelphia | San Antonio | Dallas |

Now if you only want to read the names, you can read the name column and ignore the age and city columns. This is called column wise data access. You are only reading the data you need.

```
[Alic][e, B][ob,C][arol][,Car][ol,D]...[;30,][25,3][5,35]...
```

The data we need:


```
[Alic][e, B][ob,C][arol][,Car][ol,D]...[;**,][**,*][*,**]...
```

Columnar data provides at least three benefits:
1. You are reading ONLY the data you need, nothing more
2. The data is laid out in a way that is more efficient for the computer to read. Particularly for non-text columns, such as integers and floats
3. Since data of the same data type is stored together, it can be compressed more efficiently.

On the other hand, if you do need to process all the data, you end up having to read multiple files, doing unnecessary joins and generally doing much more book keeping. Additionally, if you are writing data, each row requires writes to multiple files - a nightmare for efficient disk access. 

## Use Parquet format (columnar!), if you have the option (skip Feather?)

Parquet and Feather are columnar storage formats that are more efficient for large datasets. They allow for faster read and write operations compared to CSV, especially for large files.

Parquet is a very popular format and is widely supported by many tools, such as DuckDB, Spark, R and others. Feather is a newer format that is also being supported more and more.

A white-board explanation will follow, but the key point is that Parquet and Feather store data in a columnar format, which allows for better compression and faster access to specific columns of data. This is especially useful when working with large datasets where you may only need to access a subset of the columns.

#### Let's create `feather` and `parquet` files to test

In [ ]:
pd.read_csv(COMPRESSED_FILE).to_feather(FEATHER_FILE)
pd.read_csv(COMPRESSED_FILE).to_parquet(PARQUET_FILE)

In [ ]:
pd.read_csv(UNCOMPRESSED_4X_FILE).to_feather(FEATHER_4X_FILE)
pd.read_csv(UNCOMPRESSED_4X_FILE).to_parquet(PARQUET_4X_FILE)

In [ ]:
# Windows users
#!dir $FILE_PATH

# Mac/Linux users
%ls -lh $FILE_PATH

Notice the difference in file size for `Taxi_Trips_-_2024_20240408.csv`, `taxi_trips.feather` and `taxi_trips.parquet`. The feather file is about 1/3 the size of the csv file, and the parquet file is about 1/4 the size of the csv file. This is a huge difference in disk space usage.

The `csv` format stores data in text, which is very convenient for humans to read, but not at all efficient for computers. 

In [ ]:
%timeit pd.read_csv(UNCOMPRESSED_FILE).head(3)
%timeit pd.read_parquet(PARQUET_FILE).head(3)
%timeit pd.read_feather(FEATHER_FILE).head(3)


#### In performance testing, _scale_ matters very much! Make sure you test with data of different sizes. A small file may not show the performance difference between formats, but a large file will.

In [ ]:
%timeit pd.read_csv(UNCOMPRESSED_4X_FILE).head(3)
%timeit pd.read_parquet(PARQUET_4X_FILE).head(3)
%timeit pd.read_feather(FEATHER_4X_FILE).head(3)


As of this writing, `feather` format is not to be preferred over `parquet`.

## Use chunking when possible to avoid loading the whole file into memory

If you have a very large file or very little memory, an obvious way to solve the problem of reading the whole file into memory is to _not_ read the whole file into memory. Pandas provides a built-in way to process  dataframe in chunks!

In [ ]:
%%time
CHUNK_SIZE = 100_000
COL_NAME = "Trip Total" # Some version of the file call it trip_total

trip_grand_total = 0
for chunk in pd.read_csv(UNCOMPRESSED_4X_FILE, chunksize=CHUNK_SIZE):
    trip_grand_total += chunk[COL_NAME].sum()

If this was a very large file, you may be sitting there, wondering if the file was actually being processed. A very useful tool for such occassions is `tqdm`

In [ ]:
#!pip install tqdm

In [ ]:
from tqdm.notebook import tqdm
#tqdm.pandas()

In [ ]:
trip_grand_total = 0
for chunk in tqdm(pd.read_csv(UNCOMPRESSED_4X_FILE, chunksize=CHUNK_SIZE)):
    trip_grand_total += chunk[COL_NAME].sum()

In [ ]:
# Save space by removing unneeded variables from memory
del chunk

## Detour: Strings and integers
Some may consider this a low level detail, but data scientist and quants can benefit immensely from understanding how computers store data. 

In [ ]:
import numpy as np

In [ ]:
np.array([1]).dtype

#### What happens if you store a number as a string in a text file vs as an integer (or float) in a binary file, such as Parquet?

In [ ]:
np.array([1]).itemsize, np.array(["1"]).itemsize

In [ ]:
np.array([100]).itemsize, np.array(["100"]).itemsize

In [ ]:
np.array([1000]).itemsize, np.array(["1000"]).itemsize

In [ ]:
np.array([1565232961]).itemsize, np.array(["1565232961"]).itemsize

In other words, the number `1565232961` in a single integer, while the string "1565232961" is 10 individual characters!
A `csv` file is nothing but strings! Can you see (one of the reasons) why a parquet file is so much smaller than a csv file?

## Use correct data types to save space (potentially LOTS of it)

Let's take another look at how much space our dataframes take (check the bottom of the listing below)

In [ ]:
big_df = pd.read_csv(UNCOMPRESSED_4X_FILE)
big_df.info(memory_usage="deep")

Let's check the memory usage of each of the columns

In [ ]:
mem_df = pd.DataFrame(round(big_df.memory_usage(deep=True, index=True) / (1024 ** 2), 2), columns=["MB"])
mem_df.sort_values(by="MB", ascending=False)

In [ ]:
mem_df.sum()

In [ ]:
big_df.shape

#### Let's combine all the information about columns: name, size, data type, uniqe values

In [ ]:
mem_df\
    .join(pd.DataFrame(big_df.dtypes, columns=["dtypes"]))\
    .join(pd.DataFrame(big_df.nunique(), columns=["nuniques"]))\
    .join(big_df[:1].T)\
    .sort_values("MB")

#### Pay attention to `Payment Type`, a big columns with only 7 unique values?
This must be a categorical variable!

In [ ]:
big_df['Payment Type'].value_counts()

### Let's change it from an `object` or a `string` to a `category` type. This will save a lot of memory, and will also make some operations faster.

In [ ]:
big_df['Payment Type'].memory_usage(deep=True) / (1024 ** 2)

In [ ]:
big_df['Payment Type'] = big_df['Payment Type'].astype('category')

In [ ]:
big_df['Payment Type'].memory_usage(deep=True) / (1024 ** 2)

#### Couldn't we test all low "cardinality" columns for this technique?
"Cardinality" is a common term in databases, referring to the number of unique values in a column

In [ ]:
#pip install hvplot
import hvplot.pandas

In [ ]:
mem_df\
    .join(pd.DataFrame(big_df.dtypes, columns=["dtypes"]))\
    .join(pd.DataFrame(big_df.nunique(), columns=["nuniques"]))\
    .join(big_df[:1].T)\
    .reset_index()\
    .hvplot.scatter(x="nuniques", y="MB", hover_cols=["index"], title="Memory Usage vs Cardinality")

But don't just blindly conver all low cardinality columns, review them manually!

### Change Timestamps from string to `datetime`

In [ ]:
mem_df\
    .join(pd.DataFrame(big_df.dtypes, columns=["dtypes"]))\
    .join(pd.DataFrame(big_df.nunique(), columns=["nuniques"]))\
    .join(big_df[:1].T)\
    .sort_values("MB")

In [ ]:
big_df['Trip End Timestamp'].memory_usage(deep=True) / (1024 ** 2)

In [ ]:
big_df['Trip End Timestamp'] = pd.to_datetime(big_df['Trip End Timestamp'], format="%m/%d/%Y %H:%M:%S %p")
big_df['Trip Start Timestamp'] = pd.to_datetime(big_df['Trip Start Timestamp'], format="%m/%d/%Y %H:%M:%S %p")

In [ ]:
big_df['Trip End Timestamp'].memory_usage(deep=True) / (1024 ** 2)

### Simply drop columns you don't need
Notice that the Dropoff/Pickup centroid locations are already encoded in lat/long columns, so let's drop them!

In [ ]:
big_df.columns

In [ ]:
big_df.drop(columns=['Dropoff Centroid  Location', 'Pickup Centroid Location'], axis=1, inplace=True)

### We have saved _lots_ of memory by using common sense techniques
And remember that we didn't even work through all low-cardinality columns! (left as an exercise for the reader)

In [ ]:
big_df.info(memory_usage="deep")

## You can also just not read irrelevant columns in the first place (to save memory, not to reduce read speed)

In [ ]:
#COLS = [col for col in big_df.columns if col not in ['Dropoff Centroid  Location', 'Pickup Centroid Location']]
COLS = ['Tips', 'Fare', 'Tolls']
COLS

If you know which columns you actually need, just pass those in explicitely

In [ ]:
COLS 
pd.read_csv(UNCOMPRESSED_FILE, usecols=COLS).head(3)

#### Let's compare memory usage

In [ ]:
pd.read_csv(UNCOMPRESSED_FILE).info(memory_usage="deep")

In [ ]:
pd.read_csv(UNCOMPRESSED_FILE, usecols=COLS).info(memory_usage="deep")

723 MB vs 19.8 MB, big difference!

#### How about load time?

In [ ]:
%timeit pd.read_csv(UNCOMPRESSED_FILE).head(3)
%timeit pd.read_csv(UNCOMPRESSED_FILE, usecols=COLS).head(3)

Is the time to load different enough to matter? How about the memory usage?

## Use `memory_map` to read source file more effeciently in low memory environments
Using memory maps is an advanced technique used by programmers who know how the operating system works. Pandas makes it available for us to us much more easily. 

In [ ]:
import psutil

# Check RAM
ram = psutil.virtual_memory()
total_ram_gb = ram.total / (1024 ** 3)

# Check file size
file_size_gb = pd.read_csv(UNCOMPRESSED_4X_FILE).memory_usage(deep=True).sum() / (1024 ** 3)

print(f"Total RAM: {total_ram_gb:.2f} GB and File Size: {file_size_gb:.2f} GB")

On a computer with lots of ram, you will not see a difference in speed. However, if the file is larger than your available RAM, this can be a useful technique.

In [ ]:
%timeit pd.read_csv(UNCOMPRESSED_4X_FILE).head(3)
%timeit pd.read_csv(UNCOMPRESSED_4X_FILE, memory_map=True).head(3)

So why not use `memory_map=True` all the time? The author is not opposed to this idea! However, this can cause problems if you are loading many files (source files may be kept open by the OS) or if you are using a file that is on a network drive (the OS may not support memory mapping for network drives).

## Non-pandas solutions

### What about using the DuckDB database? It uses more cores on your computer!

In [ ]:
#pip install duckdb

In [ ]:
import duckdb

In [ ]:
%timeit duckdb.sql(f"SELECT * FROM '{UNCOMPRESSED_FILE}'").df().head(3)
%timeit duckdb.sql(f"SELECT {','.join(COLS)} FROM '{UNCOMPRESSED_FILE}'").df().head(3)

DuckDB is _much_ faster!  
**Suggestion**: When running these tests, open up your activity monitor and see _how many cpu cores are active during each test_. 

**Suggestion:** You should use databases as much as possible. If your data is coming from  your company's database, you should try to do as much computations, particularly joins, aggregations and filtering on the server. Databases, particularly data-warehouses and time series databases are designed to work on extremely large amounts of data. Databases are often run on gigantic machines, often distributed and cloud  services like BigQuery or Snowflake are essentially infinite is scale (from your perspective)

### Reduce input data: either sample or limit using domain knowledge

Your data is so large that it is hard to manage. Perhaps you can load it, but it could be computation that takes hours to complete (such as training a model). There are two obvious solutions that people often miss:

1. Use your domain knowledge to reduce data
2. Randomly sample data

#### Reduce data based on domain knowledge
Say you are building a model to predict tomorrow's opening price. Your training data has billions of trades and quotes for many weeks for all stocks.

Some obvious solutions are to build your model on the top 50 stocks, rather than the thousands of stocks traded in the US. Or perhaps it is better for your model to _skip_ the top 50 stocks, which are responsible for overwhelming majority of the trade and quote data. This way you can indeed build yoru model on thousands of lower volume stocks. Perhaps you reduce the training data from weeks to days. The correct answer depends on your goals.

#### Reduce data using sampling
Data scientists and quants often think in terms of which library function to  use to load large data and forget the most basic tool in their arsenal: sampling!

If you are building a model to predict customer churn, instead of loading millions of customers,  randomly sample from the data and  load only 10k or 100k or whatever number is appropriate. This will let you do a sanity check on your model: is there even a signal for you to optimize?

Here is some sample bash code to get you started (note this only works on text files, not binary formats like parquet):

```bash
head -n 1 big.csv > sample.csv                     # extracts the header (column names) from the source file
awk 'NR > 1 && rand() < 0.1' big.csv >> sample.csv # Randomly samples 10% of data (not including the header)
```

**IMPORTANT** Often it is good to reduce data to iterate faster whle you discover the right algorithm, model or technique. It is generally better to work through tens of variations quickly, than waiting for hours for each experiment to complete. 

### If the data doesn't fit in a computer, use multiple computers (clusters!)

At one point, no matter what technique you use, you simply can't process the data in one machine. In that scenario, you have to distribute your code across a cluster of machines. This is what tools like `pyspark` and `dask` provide. Distributed computing is very useful, sometimes necessary, but out of scope for this lecture